::: {.sb-lesson-links}
[Exercises →](../exercises/topic-modeling.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/05-topic-modeling.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-workflow.ipynb){download="lesson-workflow.ipynb"}  
Data: [newsgroups.csv](../downloads/data/newsgroups.csv){download="newsgroups.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

The full topic modeling workflow on 2,400 real messages: clean the texts, lemmatize them with spaCy, build the TF-IDF matrix, run `TruncatedSVD`, then read the topics and place the documents in the topic space.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import spacy
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

## The data

The *20 Newsgroups* dataset (collected by Ken Lang) contains messages posted in the early 1990s on 20 Usenet discussion groups. We use 400 messages from each of 6 groups: computer graphics, cars, hockey, medicine, space, and Middle East politics. The headers (sender, subject...), the signatures and the quoted replies have already been removed.

The `newsgroup` column is the group where the message was posted. **LSA will not use it**: we only keep it to check, at the end, whether the topics make sense.

In [ ]:
news = pd.read_csv("../data/newsgroups.csv")
print(news.shape)
print(news["newsgroup"].value_counts())
print("\nMessage 64:", news.loc[64, "text"])

(2400, 2)
newsgroup
sci.space                400
comp.graphics            400
rec.sport.hockey         400
sci.med                  400
talk.politics.mideast    400
rec.autos                400
Name: count, dtype: int64

Message 64: The Orion concept as actually proposed (as opposed to the way it has been somewhat misrepresented in some fiction) included wrapping a thick layer of reaction mass -- probably plastic of some sort -- around each bomb. The bomb vaporizes the reaction mass, and it's that which transfers momentum to the pusher plate.


## Cleaning and lemmatization

The steps of the NLP section:

1. **Lowercase and keep only the letters** (no digits, no punctuation).
2. **Lemmatize and remove the stop words** with spaCy. We keep the lemmas of 3 letters or more.

In [ ]:
news["clean"] = news["text"].str.lower().str.replace(r"[^a-z]+", " ", regex=True).str.strip()
print("Message 64, cleaned:", news.loc[64, "clean"])

Message 64, cleaned: the orion concept as actually proposed as opposed to the way it has been somewhat misrepresented in some fiction included wrapping a thick layer of reaction mass probably plastic of some sort around each bomb the bomb vaporizes the reaction mass and it s that which transfers momentum to the pusher plate


spaCy runs a whole pipeline on each text (tagger, parser, named entities...). We only need the lemmas, so we **disable** the parser and the named entity recognizer, and we send the texts in batches with `nlp.pipe`: both make the processing much faster. It still takes a little while on 2,400 messages.

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

lemmas = []
for doc in nlp.pipe(news["clean"], batch_size=200):
    lemmas.append(" ".join(token.lemma_ for token in doc if not token.is_stop and len(token) > 2))
news["lemmas"] = lemmas
print("Message 64, lemmatized:", news.loc[64, "lemmas"])

Message 64, lemmatized: orion concept actually propose oppose way somewhat misrepresented fiction include wrap thick layer reaction mass probably plastic sort bomb bomb vaporize reaction mass transfer momentum pusher plate


## The TF-IDF matrix

Two parameters of `TfidfVectorizer` remove the words that can't make a good topic:

* `min_df=5`: drop the words found in fewer than 5 messages (typos, rare names). A word used once can't link two documents.
* `max_df=0.5`: drop the words found in more than half of the messages. They are everywhere, so they don't tell topics apart.

In [ ]:
print("Vocabulary without limits:", len(TfidfVectorizer().fit(news["lemmas"]).vocabulary_), "words")

vectorizer = TfidfVectorizer(min_df=5, max_df=0.5)
X = vectorizer.fit_transform(news["lemmas"])
words = vectorizer.get_feature_names_out()
print("Vocabulary with min_df=5, max_df=0.5:", X.shape[1], "words")
print(f"TF-IDF matrix: {X.shape[0]} x {X.shape[1]}, {1 - X.nnz / (X.shape[0] * X.shape[1]):.2%} of zeros")

Vocabulary without limits: 16227 words
Vocabulary with min_df=5, max_df=0.5: 3573 words
TF-IDF matrix: 2400 x 3573, 98.99% of zeros


**What this shows:** the limits divide the vocabulary by more than 4, from 16,227 to 3,573 words: most words of a corpus are rare. The matrix is still 99.0% zeros, and stays sparse.

## Truncated SVD

We ask for 10 topics. `fit_transform` returns the coordinates of the 2,400 messages on the 10 topics ($U_k\Sigma_k$), `components_` the weight of the 3,573 words in each topic ($V_k^\top$).

In [ ]:
svd = TruncatedSVD(n_components=10, random_state=0)
doc_topics = svd.fit_transform(X)
print("documents x topics:", doc_topics.shape, "  topics x words:", svd.components_.shape)

topics = pd.DataFrame({
    "largest weights (+)": [", ".join(words[np.argsort(row)[::-1][:8]]) for row in svd.components_],
    "smallest weights (-)": [", ".join(words[np.argsort(row)[:5]]) for row in svd.components_],
}, index=[f"topic {j + 1}" for j in range(10)])
topics

documents x topics: (2400, 10)   topics x words: (10, 3573)


,largest weights (+),smallest weights (-)
topic 1,"know, like, think, game, don, car, good, year","subsystem, characteristic, counterpart, israel..."
topic 2,"game, team, play, hockey, espn, player, fan, win","file, car, image, program, graphic"
topic 3,"israel, arab, jews, israeli, jewish, people, k...","file, image, format, program, graphic"
topic 4,"car, engine, dealer, buy, new, oil, drive, price","file, israel, arab, image, format"
topic 5,"car, file, israel, arab, jews, format, game, i...","space, cause, effect, food, doctor"
topic 6,"team, player, play, space, good, nhl, league, ...","game, espn, doctor, cause, effect"
topic 7,"space, game, espn, orbit, shuttle, launch, moo...","player, team, good, msg, play"
topic 8,"file, format, image, problem, tiff, convert, g...","graphic, thank, card, look, information"
topic 9,"israel, image, arab, color, cause, problem, li...","don, think, muslim, file, group"
topic 10,"card, color, bit, muslim, driver, mode, image,...","space, thank, look, file, arab"


**What this shows:** read each topic from both ends.

* **Topic 1** is the "average message": *know, like, think, game, don, car...*, common words of every group. It is the "general size" topic of the previous lesson.
* **Topic 2**: hockey (*game, team, play, hockey, espn, player, fan*) against graphics and cars (*file, car, image, program*).
* **Topic 3**: Middle East politics (*israel, arab, jews, israeli, jewish*) against graphics.
* **Topic 4**: cars (*car, engine, dealer, buy, oil, drive, price*).
* **Topic 7**: space (*space, orbit, shuttle, launch, moon, mission*), mixed with two hockey words (*game, espn*).
* **Topic 8**: graphic file formats (*file, format, image, tiff, convert, gif*), a sub-theme of the graphics group.
* **Topic 10**: graphics cards (*card, color, bit, driver, mode*), mixed with *muslim*.

Topics 5, 6 and 9 are mixed (topic 6: *team, player, nhl, league* together with *space, orbit*). **No topic is about medicine**: the sci.med messages use varied, everyday words. *doctor* only appears on the negative side of topics 5 and 6.

#### See it with data 🔬: topic 1 is the average message

In the previous lesson, the first topic pointed towards the average document. Let's check on the real corpus.

In [ ]:
average_doc = np.asarray(X.mean(axis=0)).ravel()
print("Messages with a positive coordinate on topic 1:", (doc_topics[:, 0] > 0).sum(), "of", len(doc_topics))
print("Cosine between topic 1 and the average message:",
      round(svd.components_[0] @ average_doc / np.linalg.norm(average_doc), 3))

Messages with a positive coordinate on topic 1: 2400 of 2400
Cosine between topic 1 and the average message: 0.996


**What this shows:** all 2,400 messages have a positive coordinate on topic 1, and topic 1 points in the same direction as the average message (cosine 0.996). It says how "ordinary" a message is, not what it talks about. Keep that in mind before you assign each message to its "strongest" topic: topic 1 would win almost every time.

## Explained variance

As in PCA, `explained_variance_ratio_` gives the share of the variance of the data kept by each topic.

In [ ]:
svd_100 = TruncatedSVD(n_components=100, random_state=0).fit(X)
ratios = svd_100.explained_variance_ratio_
print("First 10 topics:", ratios[:10].round(4))
print(f"Total kept with 10 topics: {svd.explained_variance_ratio_.sum():.1%}, with 100 topics: {ratios.sum():.1%}")

fig = go.Figure()
fig.add_trace(go.Bar(x=np.arange(1, 101), y=ratios, name="each topic"))
fig.add_trace(go.Scatter(x=np.arange(1, 101), y=ratios.cumsum(), name="cumulative", yaxis="y2"))
fig.update_layout(title="Explained variance of the topics (TF-IDF matrix of 3,573 words)",
                  xaxis_title="topic", yaxis_title="share of the variance", height=400,
                  yaxis2=dict(title="cumulative share", overlaying="y", side="right", rangemode="tozero"))
fig.show()

First 10 topics: [0.0024 0.0081 0.0066 0.0061 0.0047 0.0039 0.0038 0.0035 0.0032 0.003 ]
Total kept with 10 topics: 4.5%, with 100 topics: 21.6%


**What this shows:** each topic keeps less than 1% of the variance (0.81% for the strongest), 10 topics keep 4.5% in total and even 100 topics only 21.6%. Text is very high-dimensional: each message has its own words. That's fine: the goal of topic modeling is not to rebuild the matrix but to find a few themes that we can read. So choose the number of topics by **reading the topics** (are they meaningful? do new ones bring new themes?), not with an "80% of the variance" rule. Notice again that topic 1 keeps less variance (0.24%) than any of the next 9 topics: it captures the average, not the differences between messages.

## The documents in the topic space

Each message now has 10 coordinates. Let's average them by newsgroup (the label LSA never saw), then plot the messages on two topics.

In [ ]:
coords = pd.DataFrame(doc_topics, columns=[f"topic {j + 1}" for j in range(10)])
coords["newsgroup"] = news["newsgroup"]
mean_coords = coords.groupby("newsgroup").mean()
fig = px.imshow(mean_coords.round(2), text_auto=True, color_continuous_scale="RdBu", color_continuous_midpoint=0,
                aspect="auto")
fig.update_layout(title="Average coordinate of the messages of each newsgroup on each topic", height=420)
fig.show()

In [ ]:
group_colors = {"comp.graphics": "#E4572E", "rec.autos": "#F2C14E", "rec.sport.hockey": "#2E7D4F",
                "sci.med": "#8E7CC3", "sci.space": "#4BA3C3", "talk.politics.mideast": "#111111"}
fig = px.scatter(coords, x="topic 2", y="topic 3", color="newsgroup", opacity=0.6, color_discrete_map=group_colors,
                 category_orders={"newsgroup": sorted(group_colors)},
                 hover_data={"newsgroup": True, "topic 2": ":.2f", "topic 3": ":.2f"})
fig.update_traces(marker_size=5)
fig.update_layout(title="2,400 messages on topics 2 and 3", height=550)
fig.show()

**What this shows:** on the heatmap, every group has about the same average on topic 1 (0.12 to 0.15): it does not separate anything. Then the hockey messages stand out on topic 2 (0.16 on average), Middle East politics on topic 3 (0.12), cars on topic 4 (0.12), space on topic 7 (0.08), while medicine is negative on topic 5 (−0.08) and graphics negative on topics 2 and 3 (−0.07 and −0.10). On topics 2 and 3, the hockey messages spread to the right, the politics messages to the top, the graphics messages go down-left, and the three other groups stay piled up near the centre: these two topics only separate three groups. The other groups need other topics, and medicine has none of its own. Topics are **directions**, not groups: to group the messages, the natural next step is to run a clustering algorithm (k-means) on these coordinates. That's what you will do in the exercise.

## Summary

* Workflow: clean (lowercase, letters only) → lemmatize and remove stop words (spaCy, `nlp.pipe`, disabled components) → `TfidfVectorizer(min_df, max_df)` → `TruncatedSVD(n_components)`.
* Read each topic with its largest **and** smallest weights in `components_`. Name the clear ones, and accept that some are mixed.
* The first topic is the average message. The explained variance of text topics is low: choose the number of topics by reading them.
* `fit_transform` places each document in the topic space, a small dense table that you can plot or cluster.

::: {.sb-next}
**[Your turn: Topic modeling exercises →](../exercises/topic-modeling.ipynb)**
:::